# run

What `process_data.yml` and the `dbt_test` job of `table_maintenance.yml` did on GitHub, in a Fabric
Python notebook: dbt-duckdb downloads the AEMO files into `nem/Files/csv` and writes the `landing`
and `mart` tables to the `nem` lakehouse through the OneLake Iceberg REST catalog (`command="run"`),
or runs the dbt tests on them (`command="test"`), or maintains the Iceberg tables
(`command="maintain"`: compaction, dangling delete files, snapshot expiry, orphan files, with the
binary `scripts/maintain/maintain` committed to the repo). The pipelines call it: `run_pipeline`
hourly (and, after its 21:00 UTC load, `maintain`), `test_pipeline` daily. Their activities hold the timeouts.

The project is the folder `scripts/deploy_fabric.py` uploaded to `nem/Files/project`: the dbt
project in its `dbt/` and the pins in `requirements.txt` (its `COMMIT` file names the commit). dbt is not preinstalled in a Fabric notebook: it is
pip-installed from the project's `requirements.txt`, the pins the GitHub runs used.


In [ ]:
# "run": the hourly load (process_data.yml); "test": dbt test (table_maintenance.yml's dbt_test);
# "maintain": the daily table maintenance (table_maintenance.yml's compact_and_expire).
command = "run"
# Files fetched per feed per run, and files turned into fact rows per feed per run. run_pipeline
# passes them: 200 and 300 on power (process_data.yml's), a few on a test workspace.
download_limit = "10"
process_limit = "10"


In [ ]:
# 1. SETTINGS: the environment process_data.yml gives dbt (AGENTS.md, "Env contract"), from
# this workspace: its id and the `nem` lakehouse's id. The OneLake token is minted before each
# dbt command (cell 4).
import json
import os
import shlex
import shutil
import subprocess
import sys
import urllib.request

import notebookutils

if command not in ("run", "test", "maintain"):
    raise ValueError(f"command must be run, test or maintain, not {command!r}")

workspace = notebookutils.runtime.context["currentWorkspaceId"]


def fabric(path):
    """One GET of the Fabric REST API for this workspace."""
    request = urllib.request.Request(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace}{path}",
        headers={"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"})
    with urllib.request.urlopen(request) as r:
        return json.loads(r.read())


found = [i["id"] for i in fabric("/lakehouses")["value"] if i["displayName"] == "nem"]
if not found:
    raise RuntimeError("no lakehouse `nem` in this workspace: deploy with deploy_fabric.yml")
lakehouse = found[0]

# The runners need curl and a CA file: the azure extension's default transport fails the
# OneLake TLS handshake.
ca = "/etc/ssl/certs/ca-certificates.crt"
if not os.path.exists(ca):
    import certifi
    ca = certifi.where()
os.environ.update({
    "ONELAKE_ENDPOINT": "https://onelake.table.fabric.microsoft.com/iceberg",
    "WAREHOUSE_PATH": f"{workspace}/{lakehouse}",
    "FILES_PATH": f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lakehouse}/Files",
    "AZURE_TRANSPORT_OPTION_TYPE": "curl",
    "CURL_CA_INFO": ca,
    "download_limit": str(download_limit),
    "process_limit": str(process_limit),
    # So dbt's log reaches the cells below line by line, not in blocks.
    "PYTHONUNBUFFERED": "1",
})
print(f"WAREHOUSE_PATH={os.environ['WAREHOUSE_PATH']}")


In [ ]:
# 2. THE PROJECT: the folder the deploy uploaded, copied to local disk, where dbt writes its
# target/ and logs/ (under dbt/).
PROJECT = "/lakehouse/default/Files/project"
if not os.path.exists(f"{PROJECT}/COMMIT"):
    raise RuntimeError(f"no dbt project in {PROJECT}: deploy with deploy_fabric.yml")
work = "/home/trusted-service-user/work" if os.path.isdir("/home/trusted-service-user/work") else "/tmp"
repo = os.path.join(work, "analytics-as-code")
shutil.rmtree(repo, ignore_errors=True)
shutil.copytree(PROJECT, repo)
print(f"{repo} @ {open(os.path.join(repo, 'COMMIT')).read().strip()}")
# The run's log, also in the lakehouse: a deploy that started the run prints its end when it
# fails (deploy_fabric.py), as the notebook's output stays in Fabric.
LOG = f"/lakehouse/default/Files/logs/{command}.log"
os.makedirs(os.path.dirname(LOG), exist_ok=True)
open(LOG, "w").close()


In [ ]:
# 3. INSTALL dbt, dbt-duckdb and duckdb, pinned in requirements.txt. dbt runs as a
# subprocess, so this kernel never imports the duckdb it replaces.
def sh(*cmd):
    """One command in the project folder, its log printed here line by line and appended to
    LOG. A failure raises, so a scheduled run goes red."""
    p = subprocess.Popen(list(cmd), cwd=repo, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    with open(LOG, "a") as log:
        log.write(f"$ {shlex.join(cmd)}\n")
        for line in p.stdout:
            print(line, end="")
            log.write(line)
    if p.wait():
        raise RuntimeError(f"{shlex.join(cmd)} failed (exit {p.returncode}); see the log above")


if command != "maintain":  # maintenance is one static binary: nothing to install
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")


In [ ]:
# 4. RUN: "run" is the two steps of process_data.yml, the landing facts and then the tables the
# semantic model reads, which are built from them; "test" is dbt test. Each dbt command gets a
# fresh OneLake token, so a long first step cannot leave the next with an expired one.
def dbt(*args):
    os.environ["ONELAKE_TOKEN"] = notebookutils.credentials.getToken("storage")
    # `python -m`, not the `dbt` executable: pip may put that outside the notebook's PATH.
    sh(sys.executable, "-m", "dbt.cli.main", *args, "--target", "prod", "--project-dir", "dbt", "--profiles-dir", "dbt")


if command == "run":
    dbt("run", "--exclude", "tag:powerbi")
    dbt("run", "--select", "tag:powerbi")
elif command == "test":
    dbt("test")


In [ ]:
# 5. MAINTAIN: scripts/maintain/maintain (Go, apache/iceberg-go; built and committed in the
# repo, nothing is compiled here), every table of `landing` and `mart` one at a time. The tool
# reads its OneLake token from a file, again for every table, and a thread here rewrites that
# file with a fresh storage token every 5 minutes, so a run longer than one token's life goes
# on. Maintenance never fails the pipeline: the tables stay as they are until tomorrow, and
# the log (Files/logs/maintain.log) and the record (Files/logs/maintenance.json) say why.
if command == "maintain":
    import base64
    import threading
    import time
    from datetime import datetime, timezone

    token_file = os.path.join(work, "onelake_token")

    def write_token():
        token = notebookutils.credentials.getToken("storage")
        with open(token_file + ".tmp", "w") as f:
            f.write(token)
        os.replace(token_file + ".tmp", token_file)
        claims = json.loads(base64.urlsafe_b64decode(token.split(".")[1] + "=="))
        line = f"token: expires {datetime.fromtimestamp(claims['exp'], timezone.utc):%H:%M:%S} UTC\n"
        print(line, end="")
        with open(LOG, "a") as log:
            log.write(line)

    def keep_token():
        while True:
            time.sleep(300)
            try:
                write_token()
            except Exception as e:  # the old file stays; the tool says when it runs out
                print(f"token refresh failed: {e}")

    write_token()
    threading.Thread(target=keep_token, daemon=True).start()

    binary = os.path.join(repo, "scripts", "maintain", "maintain")
    os.chmod(binary, 0o755)
    record = os.path.join(work, "maintenance.json")
    os.environ.update({
        "WS_ID": workspace,
        "LH_ID": lakehouse,
        "ONELAKE_TOKEN_FILE": token_file,
        "RUN_ID": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        # Under the activity's 1:30 timeout: the tool stops starting new tables at the budget.
        "COMPACT_BUDGET_MINUTES": "70",
        "EXPIRE_OLDER_THAN_DAYS": "1",
        "ORPHAN_OLDER_THAN_DAYS": "3",
    })
    try:
        sh(binary, record)
    except RuntimeError as e:
        print(f"WARNING: {e}")
    finally:
        if os.path.exists(record):
            shutil.copy(record, "/lakehouse/default/Files/logs/maintenance.json")
